# Ingest Dimension Data into Bronze Layer

## Overview

This notebook ingests raw dimension datasets from the Unity Catalog Volume into the Bronze layer of the Medallion Architecture.

The ingestion process performs the following tasks:
- Reads raw CSV files from the Raw Landing area.
- Applies predefined schemas to ensure data consistency.
- Adds ingestion metadata for data lineage and auditing.
- Validates the loaded data.
- Stores the data as Delta tables in the Bronze schema.

The Bronze layer preserves the original source data with minimal transformation, providing a reliable foundation for downstream Silver layer processing.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
from pyspark.sql import functions as F

## Catalog Configuration

Defines the Unity Catalog used throughout the notebook. All source data is read from Volumes within this catalog, and the resulting Delta tables are created in the corresponding Bronze schema.

In [0]:
catalog_name = "ecommerce"

## Brand

### Purpose
This section ingests the **Brand** dimension data from the Raw Landing Volume into the Bronze layer.

### Processing Steps
- Defines the schema for the Brand dataset.
- Reads the raw CSV files from the Unity Catalog Volume.
- Adds metadata columns (`_source_file` and `_ingest_time`) to support data lineage and auditing.
- Validates the loaded records.
- Writes the data to the `brz_brands` Delta table.

### Output
**Bronze Table:** `bronze.brz_brands`

In [0]:
# Brand Schema Definition
brand_schema = StructType([
    StructField("brand_code", StringType(), False),
    StructField("brand_name", StringType(), True),
    StructField("category_code", StringType(), True),
])

# Read Raw Data
raw_data_path = f"/Volumes/{catalog_name}/raw/raw_landing/brands/*.csv"

df_raw = spark.read.option("header", True).option("delimiter", ",").schema(brand_schema).csv(raw_data_path)

# Add Ingestion Metadata Columns
df_raw = df_raw.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("_ingest_time", F.current_timestamp())

# Validate Data
display(df_raw.limit(5))

# Write Data to Bronze Layer (catalog: ecommerce, schema: bronze, table: brz_brands)
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_brands")

brand_code,brand_name,category_code,_source_file,_ingest_time
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:41.844Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:41.844Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:41.844Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:41.844Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:41.844Z


## Category

### Purpose
This section ingests the **Category** dimension data from the Raw Landing Volume into the Bronze layer.

### Processing Steps
- Defines the schema for the Category dataset.
- Reads the raw CSV files from the Unity Catalog Volume.
- Adds metadata columns (`_source_file` and `_ingest_time`) for data lineage and auditing.
- Validates the loaded records.
- Writes the data to the `brz_category` Delta table.

### Output
**Bronze Table:** `bronze.brz_category`

In [0]:
# Category Schema Definition
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True)
])

# Load data using the schema defined
raw_data_path = f"/Volumes/{catalog_name}/raw/raw_landing/category/*.csv" 

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(category_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("_ingest_time", F.current_timestamp())

# Validate Data
display(df.limit(5))

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_category)
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_category")    

brand_code,brand_name,category_code,_source_file,_ingest_time
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:44.215Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:44.215Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:44.215Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:44.215Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:44.215Z


## Products

### Purpose
This section ingests the **Product** dimension data from the Raw Landing Volume into the Bronze layer.

### Processing Steps
- Defines the schema for the Product dataset.
- Reads the raw CSV files from the Unity Catalog Volume.
- Adds metadata columns (`_source_file` and `_ingest_time`) for data lineage and auditing.
- Validates the loaded records.
- Writes the data to the `brz_products` Delta table.

### Output
**Bronze Table:** `bronze.brz_products`

In [0]:
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),  #datatype is string due to incoming data contain anamolies
    StructField("length_cm", StringType(), True),     #datatype is string due to incoming data contain anamolies
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False)
])

# Load data using the schema defined
raw_data_path = f"/Volumes/{catalog_name}/raw/raw_landing/products/*.csv" 

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(products_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("_ingest_time", F.current_timestamp())

# Validate Data
display(df.limit(5))

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_products)
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_products")  

brand_code,brand_name,category_code,_source_file,_ingest_time
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:48.315Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:48.315Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:48.315Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:48.315Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:48.315Z


## Customers

### Purpose
This section ingests the **Customer** dimension data from the Raw Landing Volume into the Bronze layer.

### Processing Steps
- Defines the schema for the Customer dataset.
- Reads the raw CSV files from the Unity Catalog Volume.
- Adds metadata columns (`_source_file` and `_ingest_time`) for data lineage and auditing.
- Validates the loaded records.
- Writes the data to the `brz_customers` Delta table.

### Output
**Bronze Table:** `bronze.brz_customers`

In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

# Load data using the schema defined
raw_data_path = f"/Volumes/{catalog_name}/raw/raw_landing/customers/*.csv" 

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(customers_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("_ingest_time", F.current_timestamp())

# Validate Data
display(df.limit(5))

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_customers)
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_customers")      

brand_code,brand_name,category_code,_source_file,_ingest_time
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:51.227Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:51.227Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:51.227Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:51.227Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:51.227Z


## Date

### Purpose
This section ingests the **Date** dimension data into the Bronze layer.

### Processing Steps
- Defines the schema for the Date dataset.
- Reads the raw CSV files from the Unity Catalog Volume.
- Adds metadata columns (`_source_file` and `_ingest_time`) for data lineage and auditing.
- Validates the loaded records.
- Writes the data to the `brz_date` Delta table.

### Output
**Bronze Table:** `bronze.brz_date`

In [0]:

# Define schema for the data file
date_schema = StructType([
    StructField("date", DateType(), True),           # Raw date in string format
    StructField("year", IntegerType(), True),          # Year
    StructField("day_name", StringType(), True),       # Day name (can be mixed case)
    StructField("quarter", IntegerType(), True),       # Quarter
    StructField("week_of_year", IntegerType(), True),  # Week of year (can be negative)
])

# Load data using the schema defined
raw_data_path = f"/Volumes/{catalog_name}/raw/raw_landing/date/*.csv" 

df_raw = spark.read.option("header", "true").option("delimiter", ",").schema(date_schema).csv(raw_data_path)

# Add metadata columns
df_raw = df_raw.withColumn("_source_file", F.col("_metadata.file_path")) \
    .withColumn("_ingest_time", F.current_timestamp())

# Validate Data
display(df.limit(5))

# Write raw data to the Bronze layer (catalog: ecommerce, schema: bronze, table: brz_calendar) 
df_raw.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_calendar")               

brand_code,brand_name,category_code,_source_file,_ingest_time
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:54.214Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:54.214Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:54.214Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:54.214Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/raw/raw_landing/brands/brands.csv,2026-07-23T06:35:54.214Z
